# 1 · Extract a character's dialogue

Open in **Google Colab**, choose **Runtime → Change runtime type → GPU**. Run installation once, then **Runtime → Restart session** and continue at configuration. Everything runs inside Colab; no paid inference API is used.

Accept the conditions for [pyannote Community-1](https://huggingface.co/pyannote/speaker-diarization-community-1), create a Hugging Face **read** token, and add it as the Colab secret `HF_TOKEN` with notebook access. This downloads model weights; the episode is processed in the Colab runtime.

First episode: upload one MP3 and provide several **start/end** samples containing only your target voice, ideally 2–8 seconds each and varied delivery. Later episodes: upload the MP3 plus `speaker_profile.json`, leave `SAMPLE_TIMESTAMPS` empty. Download the ZIP before the runtime expires.

Diarization and speaker matching are estimates. Review the speaker table and audio previews before export. A profile describes a performed voice; a different character, whisper, or scream may need additional samples. Clips retain the original mixture, including music and simultaneous speakers: this notebook does not perform source separation.

Keep generated archives, embeddings, and transcripts private. Before publishing a notebook, clear all outputs and remove personal inputs/tokens. Publishing code alone does not determine whether a particular use of source media is permitted.

In [ ]:
# Run ONLY in Colab. Restart the session after this cell, then skip this cell.
import sys

if "google.colab" not in sys.modules:
    raise RuntimeError("Open this notebook in Google Colab before installing dependencies.")

%pip install -q "torch==2.8.0" "torchvision==0.23.0" "torchaudio==2.8.0" "torchcodec==0.7.0" "pyannote.audio==4.0.4" "soundfile==0.13.1"

## Configure

IDs become metadata and directory names: use letters, digits, `_`, or `-`. Enter the credited actor yourself; the models cannot determine an actor's real-world identity. Times can be seconds, `MM:SS.sss`, or `HH:MM:SS.sss`.

`SPEAKER_THRESHOLD` is an uncalibrated cosine-similarity cutoff, **not a probability**. Inspect near matches and adjust it for your material. More than one cluster can match the same target. Override incorrect selections with `SPEAKER_OVERRIDE` after examining the results.

In [ ]:
from typing import TypeAlias

TimestampValue: TypeAlias = str | int | float

ACTOR_ID: str = "actor_x"
CHARACTER_ID: str = "character_x"
SERIES_ID: str = "series_x"
EPISODE_ID: str = "episode_01"
LANGUAGE: str = "ja"  # Spoken language, including the dub language when applicable.

INPUT_AUDIO_FILE: str = ""  # Empty opens Colab's upload dialog.
PROFILE_FILE: str = ""      # Empty auto-detects an uploaded profile JSON.
SAMPLE_TIMESTAMPS: list[tuple[TimestampValue, TimestampValue]] = [
    # ("00:01:15.679", "00:01:18.129"),
    # ("00:05:20.000", "00:05:24.000"),
]

SPEAKER_THRESHOLD: float = 0.55
SPEAKER_OVERRIDE: list[str] | None = None  # None = automatic; e.g. ["SPEAKER_02", "SPEAKER_05"].
SILENCE_TOLERANCE_SECONDS: float = 2.0
BOUNDARY_PADDING_SECONDS: float = 0.08
INCLUDE_OVERLAPPING_SPEECH: bool = True  # Retain and flag; False removes overlapping portions.
MAX_SPEAKERS: int | None = None              # Set only if you have a useful upper bound.
EMBEDDING_SECONDS_PER_SPEAKER: float = 60.0
PREVIEW_CLIPS: int = 6
DIARIZATION_SEGMENTATION_BATCH_SIZE: int = 16
DIARIZATION_EMBEDDING_BATCH_SIZE: int = 16
EXPORT_WORKERS: int = 2  # Parallel FLAC encoding and hashing on the CPU; 1 disables concurrency.

DIARIZATION_MODEL: str = "pyannote/speaker-diarization-community-1"
DIARIZATION_REVISION: str = "3533c8cf8e369892e6b79ff1bf80f7b0286a54ee"
EMBEDDING_MODEL: str = "pyannote/wespeaker-voxceleb-resnet34-LM"
EMBEDDING_REVISION: str = "837717ddb9ff5507820346191109dc79c958d614"

In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import re
import shutil
import subprocess
import uuid
import zipfile
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, TypeAlias, TypedDict
from time import perf_counter

os.environ["PYANNOTE_METRICS_ENABLED"] = "0"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import numpy as np
from numpy.typing import ArrayLike, NDArray
import pandas as pd
import soundfile as sf
import torch
from google.colab import files, userdata
from IPython.display import Audio, display
from pyannote.audio import Inference, Model, Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook
from tqdm.auto import tqdm

JsonObject: TypeAlias = dict[str, Any]
FloatArray: TypeAlias = NDArray[np.float32]
Interval: TypeAlias = tuple[float, float]


class SpeakerTurn(TypedDict):
    start: float
    end: float
    speaker: str


class MergedSegment(TypedDict):
    start: float
    end: float
    speech_intervals: list[Interval]


if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime, then restart the session.")
if not shutil.which("ffmpeg"):
    raise RuntimeError("This runtime has no ffmpeg. Use a standard Colab runtime.")

for field, value in {
    "ACTOR_ID": ACTOR_ID, "CHARACTER_ID": CHARACTER_ID,
    "SERIES_ID": SERIES_ID, "EPISODE_ID": EPISODE_ID,
}.items():
    if not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError(f"{field} must contain only letters, digits, underscores, or hyphens.")
if not -1 <= SPEAKER_THRESHOLD <= 1:
    raise ValueError("SPEAKER_THRESHOLD must be between -1 and 1.")
if min(SILENCE_TOLERANCE_SECONDS, BOUNDARY_PADDING_SECONDS) < 0:
    raise ValueError("Gap tolerance and padding must be nonnegative.")
if EMBEDDING_SECONDS_PER_SPEAKER < 1:
    raise ValueError("EMBEDDING_SECONDS_PER_SPEAKER must be at least 1.")
if min(DIARIZATION_SEGMENTATION_BATCH_SIZE, DIARIZATION_EMBEDDING_BATCH_SIZE, EXPORT_WORKERS) < 1:
    raise ValueError("Diarization batch sizes and export worker count must be positive.")

DEVICE: torch.device = torch.device("cuda")
SAMPLE_RATE: int = 16000
print("GPU:", torch.cuda.get_device_name(0))


def write_json(path: Path, value: JsonObject) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def parse_time(value: TimestampValue) -> float:
    if isinstance(value, (int, float)):
        seconds = float(value)
    else:
        parts = str(value).strip().split(":")
        if not 1 <= len(parts) <= 3:
            raise ValueError(f"Invalid timestamp: {value}")
        numbers = [float(part) for part in parts]
        if any(number < 0 for number in numbers):
            raise ValueError(f"Negative timestamp: {value}")
        if len(numbers) > 1 and any(number >= 60 for number in numbers[1:]):
            raise ValueError(f"Minutes/seconds outside timestamp range: {value}")
        seconds = sum(number * 60 ** index for index, number in enumerate(reversed(numbers)))
    if not math.isfinite(seconds) or seconds < 0:
        raise ValueError(f"Invalid timestamp: {value}")
    return seconds


def union_intervals(intervals: Iterable[Interval]) -> list[Interval]:
    merged: list[Interval] = []
    for start, end in sorted(intervals):
        if end <= start:
            continue
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((float(start), float(end)))
    return merged


def subtract_intervals(start: float, end: float, blockers: list[Interval]) -> list[Interval]:
    pieces: list[Interval] = []
    cursor = start
    for left, right in blockers:
        if right <= cursor:
            continue
        if left >= end:
            break
        if left > cursor:
            pieces.append((cursor, min(left, end)))
        cursor = max(cursor, right)
        if cursor >= end:
            break
    if cursor < end:
        pieces.append((cursor, end))
    return pieces


def overlap_seconds(start: float, end: float, intervals: list[Interval]) -> float:
    return sum(max(0.0, min(end, right) - max(start, left)) for left, right in intervals)

## Upload and decode

Upload exactly one audio file. To reuse a profile, include its standalone JSON in the same upload, or set `PROFILE_FILE` to an existing Colab path. Samples supplied alongside a compatible profile add exemplars to it; the source profile file is left untouched.

In [ ]:
uploaded_names: list[str] = []
if not INPUT_AUDIO_FILE:
    uploaded = files.upload()
    uploaded_names: list[str] = list(uploaded)
    audio_names: list[str] = [name for name in uploaded_names if Path(name).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".ogg"}]
    if len(audio_names) != 1:
        raise ValueError("Upload exactly one episode audio file.")
    INPUT_AUDIO_FILE: str = audio_names[0]
    del uploaded  # Release the extra in-memory upload copy.

if not PROFILE_FILE:
    profile_names: list[str] = [name for name in uploaded_names if Path(name).suffix.lower() == ".json"]
    if len(profile_names) > 1:
        raise ValueError("Choose one speaker profile with PROFILE_FILE.")
    if profile_names:
        PROFILE_FILE: str = profile_names[0]
if not SAMPLE_TIMESTAMPS and not PROFILE_FILE:
    raise ValueError("Enter SAMPLE_TIMESTAMPS for your first episode, or upload a saved profile.")

audio_path: Path = Path(INPUT_AUDIO_FILE)
source_sha256: str = file_sha256(audio_path)
run_id: str = uuid.uuid4().hex[:12]
run_root: Path = Path("/content/voice-stat") / SERIES_ID / EPISODE_ID / CHARACTER_ID / run_id
run_root.mkdir(parents=True, exist_ok=False)
bundle_root: Path = run_root / "extraction"
bundle_root.mkdir()
(bundle_root / "segments").mkdir()
(bundle_root / "profile").mkdir()
(bundle_root / "review").mkdir()

decoded_path: Path = run_root / "episode_16k_mono.wav"
command: list[str] = ["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",
           "-i", str(audio_path), "-vn", "-ac", "1", "-ar", str(SAMPLE_RATE),
           "-c:a", "pcm_s16le", str(decoded_path)]
result: subprocess.CompletedProcess[str] = subprocess.run(command, capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(f"Audio conversion failed: {result.stderr[-2000:]}")
audio: FloatArray
rate: int
audio, rate = sf.read(decoded_path, dtype="float32")
duration = len(audio) / rate
if rate != SAMPLE_RATE or audio.ndim != 1 or duration <= 0:
    raise ValueError("Expected a nonempty 16 kHz mono decoded episode.")

samples: list[Interval] = []
for start_value, end_value in SAMPLE_TIMESTAMPS:
    start, end = parse_time(start_value), parse_time(end_value)
    if not 0 <= start < end <= duration:
        raise ValueError(f"Sample {start_value}–{end_value} is outside the {duration:.3f}s episode.")
    if end - start < 1:
        raise ValueError("Choose voice samples at least one second long; 2–8 seconds is preferable.")
    samples.append((start, end))
    display(Audio(audio[round(start * rate):round(end * rate)], rate=rate))

print(f"Episode duration: {duration / 60:.2f} minutes; source hash: {source_sha256[:12]}")

## Diarize the whole episode

This finds estimated speaker turns, including overlapping turns. It preserves fractional-second timestamps; boundaries are model estimates. If the result looks wrong, change `MAX_SPEAKERS` or improve your samples. The original actor labels are provided by you, not inferred by the diarizer.

The two `DIARIZATION_*_BATCH_SIZE` settings process multiple audio windows per GPU batch. Start with 16; try 32 if memory permits, or reduce to 8/4 after a memory error. The separate reusable-profile matching step processes its variable-length samples sequentially.

In [ ]:
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception as error:
    raise RuntimeError("Add the HF_TOKEN read token to Colab Secrets and enable notebook access.") from error

pipeline = Pipeline.from_pretrained(
    DIARIZATION_MODEL, revision=DIARIZATION_REVISION, token=hf_token,
)
if pipeline is None:
    raise RuntimeError("Model access failed. Accept Community-1 conditions and check your HF token.")
pipeline.to(DEVICE)
pipeline.segmentation_batch_size = DIARIZATION_SEGMENTATION_BATCH_SIZE
pipeline.embedding_batch_size = DIARIZATION_EMBEDDING_BATCH_SIZE
kwargs: dict[str, int] = {} if MAX_SPEAKERS is None else {"max_speakers": MAX_SPEAKERS}
diarization_started: float = perf_counter()
with ProgressHook() as hook:
    diarization_output = pipeline(
        {"waveform": torch.from_numpy(audio).unsqueeze(0), "sample_rate": rate},
        hook=hook, **kwargs,
    )

turns: list[SpeakerTurn] = sorted([
    {"start": max(0.0, float(turn.start)), "end": min(duration, float(turn.end)), "speaker": speaker}
    for turn, speaker in diarization_output.speaker_diarization
    if min(duration, turn.end) > max(0.0, turn.start)
], key=lambda turn: (turn["start"], turn["end"], turn["speaker"]))
if not turns:
    raise RuntimeError("No speech was detected in the episode.")
write_json(bundle_root / "diarization.json", {"schema_version": 1, "turns": turns})

del pipeline, diarization_output
gc.collect()
torch.cuda.empty_cache()
print(f"Detected {len(turns)} turns across {len({turn['speaker'] for turn in turns})} clusters "
      f"in {perf_counter() - diarization_started:.1f}s (excluding model download/load).")

In [ ]:
profile: JsonObject | None = None
if PROFILE_FILE:
    profile: JsonObject | None = json.loads(Path(PROFILE_FILE).read_text(encoding="utf-8"))
    if profile.get("schema_version") != 1 or profile.get("kind") != "voice_stat_speaker_profile":
        raise ValueError("This is not a compatible voice-stat speaker profile.")
    if profile["actor_id"] != ACTOR_ID or profile["character_id"] != CHARACTER_ID:
        raise ValueError("Set ACTOR_ID and CHARACTER_ID to the saved profile's IDs.")
    if profile["embedding_model"] != EMBEDDING_MODEL:
        raise ValueError("Saved profile uses a different embedding model.")
    EMBEDDING_REVISION: str = profile["embedding_revision"]  # Keep the embedding space identical.

embedding_model = Model.from_pretrained(
    EMBEDDING_MODEL, revision=EMBEDDING_REVISION, token=hf_token,
)
if embedding_model is None:
    raise RuntimeError("Speaker embedding model could not be loaded.")
embedding_inference = Inference(embedding_model, window="whole")
embedding_inference.to(DEVICE)
del hf_token


def normalize_embedding(vector: ArrayLike) -> FloatArray:
    embedding: FloatArray = np.asarray(vector, dtype=np.float32).reshape(-1)
    norm = float(np.linalg.norm(embedding))
    if not np.isfinite(embedding).all() or norm == 0:
        raise ValueError("Invalid embedding; choose a longer, clearer voice sample.")
    return embedding / norm


def embed_interval(start: float, end: float) -> FloatArray:
    clip: FloatArray = audio[round(start * rate):round(end * rate)]
    with torch.inference_mode():
        vector = embedding_inference({"waveform": torch.from_numpy(clip).unsqueeze(0), "sample_rate": rate})
    return normalize_embedding(vector)


exemplars: list[FloatArray] = [] if profile is None else [normalize_embedding(vector) for vector in profile["exemplars"]]
for start, end in tqdm(samples, desc="Profile samples"):
    # Bound inference size without discarding any of the supplied sample.
    count = max(1, math.ceil((end - start) / 8))
    edges = np.linspace(start, end, count + 1)
    for left, right in zip(edges[:-1], edges[1:]):
        exemplars.append(embed_interval(float(left), float(right)))
reference_vectors: FloatArray = np.stack(exemplars)
if profile and reference_vectors.shape[1] != profile["embedding_dimension"]:
    raise ValueError("Saved profile embedding dimensions do not match.")
reference_centroid: FloatArray = normalize_embedding(reference_vectors.mean(axis=0))
if profile is None:
    profile: JsonObject | None = {
        "schema_version": 1, "kind": "voice_stat_speaker_profile",
        "profile_id": uuid.uuid4().hex,
        "actor_id": ACTOR_ID, "character_id": CHARACTER_ID,
        "embedding_model": EMBEDDING_MODEL, "embedding_revision": EMBEDDING_REVISION,
        "created_utc": datetime.now(timezone.utc).isoformat(), "sample_sources": [],
    }
profile.update({
    "embedding_dimension": int(reference_vectors.shape[1]),
    "exemplars": reference_vectors.tolist(), "centroid": reference_centroid.tolist(),
})
if samples:
    profile["sample_sources"].append({"source_sha256": source_sha256, "intervals_seconds": samples})
write_json(bundle_root / "profile" / "speaker_profile.json", profile)

cluster_rows: list[JsonObject] = []
speaker_examples: dict[str, Interval] = {}
matching_started: float = perf_counter()
for speaker in tqdm(sorted({turn["speaker"] for turn in turns}), desc="Match speaker clusters"):
    speaker_turns: list[SpeakerTurn] = [turn for turn in turns if turn["speaker"] == speaker]
    other_intervals: list[Interval] = union_intervals([(turn["start"], turn["end"]) for turn in turns if turn["speaker"] != speaker])
    clean_pieces: list[Interval] = []
    for turn in speaker_turns:
        clean_pieces.extend(subtract_intervals(turn["start"], turn["end"], other_intervals))
    clean_pieces: list[Interval] = union_intervals(clean_pieces)
    windows: list[Interval] = []
    for left, right in clean_pieces:
        count = max(1, math.ceil((right - left) / 8))
        edges = np.linspace(left, right, count + 1)
        windows.extend((float(a), float(b)) for a, b in zip(edges[:-1], edges[1:]) if b - a >= 1)
    windows.sort(key=lambda window: window[1] - window[0], reverse=True)
    vectors: list[FloatArray] = []
    weights: list[float] = []
    used_seconds = 0.0
    for left, right in windows:
        remaining = EMBEDDING_SECONDS_PER_SPEAKER - used_seconds
        if remaining < 1:
            break
        right = min(right, left + remaining)
        vectors.append(embed_interval(left, right))
        weights.append(right - left)
        used_seconds += right - left
    centroid_similarity: float | None = None
    exemplar_similarity: float | None = None
    score: float | None = None
    longest_turn = max(speaker_turns, key=lambda turn: turn["end"] - turn["start"])
    speaker_examples[speaker] = (longest_turn["start"], min(longest_turn["end"], longest_turn["start"] + 8))
    if vectors:
        cluster_centroid = normalize_embedding(np.average(np.stack(vectors), axis=0, weights=weights))
        centroid_similarity = float(cluster_centroid @ reference_centroid)
        exemplar_similarity = float(np.max(reference_vectors @ cluster_centroid))
        score = max(centroid_similarity, exemplar_similarity)
        speaker_examples[speaker] = windows[0]
    # Samples identify known clusters only when one cluster occupies >=80% and others <=10%.
    seed_hits = 0
    for start, end in samples:
        own_intervals = union_intervals([(turn["start"], turn["end"]) for turn in speaker_turns])
        own_fraction = overlap_seconds(start, end, own_intervals) / (end - start)
        other_fraction = overlap_seconds(start, end, other_intervals) / (end - start)
        if own_fraction >= 0.8 and other_fraction <= 0.1:
            seed_hits += 1
    cluster_rows.append({
        "speaker": speaker, "similarity": score, "centroid_similarity": centroid_similarity,
        "exemplar_similarity": exemplar_similarity, "sample_hits": seed_hits,
        "embedded_seconds": used_seconds,
        "speech_seconds": sum(right - left for left, right in union_intervals([(turn["start"], turn["end"]) for turn in speaker_turns])),
        "automatic_match": bool(seed_hits or (score is not None and score >= SPEAKER_THRESHOLD)),
    })

cluster_table: pd.DataFrame = pd.DataFrame(cluster_rows).sort_values("similarity", ascending=False, na_position="last")
cluster_table.to_csv(bundle_root / "review" / "speaker_matches.csv", index=False)
display(cluster_table)
del embedding_inference, embedding_model
gc.collect()
torch.cuda.empty_cache()
print(f"Speaker-cluster matching: {perf_counter() - matching_started:.1f}s.")

In [ ]:
# Preview any cluster without rerunning GPU matching. Empty = the closest matches.
PREVIEW_SPEAKERS: list[str] = []  # e.g. ["SPEAKER_02", "SPEAKER_05"]
preview_speakers = PREVIEW_SPEAKERS or cluster_table.head(PREVIEW_CLIPS)["speaker"].tolist()
for speaker in preview_speakers:
    if speaker not in speaker_examples:
        raise ValueError(f"Unknown speaker ID: {speaker}")
    left, right = speaker_examples[speaker]
    print(speaker, "similarity:", cluster_table.loc[cluster_table["speaker"] == speaker, "similarity"].iloc[0])
    display(Audio(audio[round(left * rate):round(right * rate)], rate=rate))

## Select, merge, and export

After listening, change `SPEAKER_OVERRIDE` below if needed. Rerun this cell and the cells beneath it after changing your selection; GPU matching does not need to run again.

Target turns separated by at most the configured gap are saved as one continuous clip, including that gap. Another detected speaker in the gap forces a split. Overlapping target turns are combined without duplicate audio. Padding stops at neighboring non-target speech. With overlap exclusion enabled, simultaneous speech is removed before merging and never reintroduced through padding or gaps.

Counts here represent merged **clips**, not grammatical sentences. Short interjections are retained. Long clips can contain several emotions; preserve this distinction when interpreting your results.

In [ ]:
# Set this here after reviewing the speaker table, if necessary.
# SPEAKER_OVERRIDE = ["SPEAKER_02"]

known_speakers: set[str] = {row["speaker"] for row in cluster_rows}
selected_speakers: set[str] = (
    {row["speaker"] for row in cluster_rows if row["automatic_match"]}
    if SPEAKER_OVERRIDE is None else set(SPEAKER_OVERRIDE)
)
if not selected_speakers <= known_speakers:
    raise ValueError(f"Unknown speaker IDs: {selected_speakers - known_speakers}")
if not selected_speakers:
    raise ValueError("No target matched. Review the table and adjust the threshold or selection.")

other_intervals: list[Interval] = union_intervals([(turn["start"], turn["end"]) for turn in turns if turn["speaker"] not in selected_speakers])
target_intervals: list[Interval] = union_intervals([(turn["start"], turn["end"]) for turn in turns if turn["speaker"] in selected_speakers])
if not INCLUDE_OVERLAPPING_SPEECH:
    target_intervals: list[Interval] = [piece for left, right in target_intervals for piece in subtract_intervals(left, right, other_intervals)]

merged_segments: list[MergedSegment] = []
for left, right in target_intervals:
    if merged_segments:
        previous = merged_segments[-1]
        gap = left - previous["end"]
        foreign_speech_in_gap = overlap_seconds(previous["end"], left, other_intervals) > 0
        if gap <= SILENCE_TOLERANCE_SECONDS and not foreign_speech_in_gap:
            previous["end"] = max(previous["end"], right)
            previous["speech_intervals"].append((left, right))
            continue
    merged_segments.append({"start": left, "end": right, "speech_intervals": [(left, right)]})

def export_segment(task: tuple[int, MergedSegment]) -> JsonObject | None:
    index, segment = task
    left, right = segment["start"], segment["end"]
    clip_start = max(0.0, left - BOUNDARY_PADDING_SECONDS)
    clip_end = min(duration, right + BOUNDARY_PADDING_SECONDS)
    # Do not pad into another clip or newly include non-target speech.
    if index > 1:
        clip_start = max(clip_start, (merged_segments[index - 2]["end"] + left) / 2)
    if index < len(merged_segments):
        clip_end = min(clip_end, (right + merged_segments[index]["start"]) / 2)
    for other_start, other_end in other_intervals:
        if other_start < left and other_end > clip_start:
            clip_start = max(clip_start, min(left, other_end))
        if other_end > right and other_start < clip_end:
            clip_end = min(clip_end, max(right, other_start))
    start_sample, end_sample = round(clip_start * rate), round(clip_end * rate)
    if end_sample <= start_sample:
        return None
    clip_start, clip_end = start_sample / rate, end_sample / rate
    identity = json.dumps([source_sha256, ACTOR_ID, CHARACTER_ID, start_sample, end_sample])
    observation_id = hashlib.sha256(identity.encode()).hexdigest()[:24]
    filename = f"segment_{index:04d}_{start_sample:010d}-{end_sample:010d}.flac"
    relative_path = f"segments/{filename}"
    sf.write(bundle_root / relative_path, audio[start_sample:end_sample], rate, subtype="PCM_16")
    flags: list[str] = []
    overlap = overlap_seconds(clip_start, clip_end, other_intervals)
    speech_seconds = sum(b - a for a, b in union_intervals(segment["speech_intervals"]))
    if overlap > 0:
        flags.append("overlapping_non_target_speech")
    if speech_seconds < 1:
        flags.append("short_speech")
    if right - left > 30:
        flags.append("long_clip_may_contain_multiple_emotions")
    return {
        "observation_id": observation_id, "actor_id": ACTOR_ID, "character_id": CHARACTER_ID,
        "series_id": SERIES_ID, "episode_id": EPISODE_ID, "language": LANGUAGE,
        "source_sha256": source_sha256, "audio_file": relative_path,
        "audio_sha256": file_sha256(bundle_root / relative_path),
        "start_seconds": clip_start, "end_seconds": clip_end,
        "speech_start_seconds": left, "speech_end_seconds": right,
        "speech_intervals": segment["speech_intervals"], "speech_seconds": speech_seconds,
        "clip_seconds": clip_end - clip_start, "overlap_seconds": overlap,
        "speaker_clusters": sorted({
            turn["speaker"] for turn in turns
            if turn["speaker"] in selected_speakers and turn["start"] < right and turn["end"] > left
        }),
        "quality_flags": flags,
    }


export_started: float = perf_counter()
with ThreadPoolExecutor(max_workers=EXPORT_WORKERS) as executor:
    exported_records = executor.map(export_segment, enumerate(merged_segments, start=1))
    records: list[JsonObject] = [
        record for record in tqdm(exported_records, total=len(merged_segments), desc="Export FLAC clips")
        if record is not None
    ]
if not records:
    raise ValueError("No target audio remains after the overlap policy.")
print(f"Audio export: {perf_counter() - export_started:.1f}s with {EXPORT_WORKERS} workers.")

# On export, only current records are included; earlier previews are not added to the ZIP.
write_json(bundle_root / "segments.json", {"schema_version": 1, "segments": records})
manifest: JsonObject = {
    "schema_version": 1, "kind": "voice_stat_extraction", "run_id": run_id,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "actor_id": ACTOR_ID, "character_id": CHARACTER_ID, "series_id": SERIES_ID,
    "episode_id": EPISODE_ID, "language": LANGUAGE,
    "source_sha256": source_sha256, "source_duration_seconds": duration,
    "sample_rate": rate, "channels": 1, "audio_format": "FLAC PCM_16",
    "models": {"diarization": {"id": DIARIZATION_MODEL, "revision": DIARIZATION_REVISION},
               "embedding": {"id": EMBEDDING_MODEL, "revision": EMBEDDING_REVISION}},
    "settings": {"speaker_threshold": SPEAKER_THRESHOLD, "selected_speakers": sorted(selected_speakers),
                 "selection_method": "manual_override" if SPEAKER_OVERRIDE is not None else "similarity_and_sample_hits",
                 "silence_tolerance_seconds": SILENCE_TOLERANCE_SECONDS,
                 "padding_seconds": BOUNDARY_PADDING_SECONDS,
                 "include_overlapping_speech": INCLUDE_OVERLAPPING_SPEECH,
                 "embedding_seconds_per_speaker": EMBEDDING_SECONDS_PER_SPEAKER,
                 "diarization_segmentation_batch_size": DIARIZATION_SEGMENTATION_BATCH_SIZE,
                 "diarization_embedding_batch_size": DIARIZATION_EMBEDDING_BATCH_SIZE,
                 "export_workers": EXPORT_WORKERS},
    "package_versions": {name: importlib.metadata.version(name) for name in ["torch", "torchaudio", "pyannote.audio", "soundfile"]},
    "segment_count": len(records), "segments_file": "segments.json",
    "profile_file": "profile/speaker_profile.json",
}
write_json(bundle_root / "manifest.json", manifest)
display(pd.DataFrame(records)[["observation_id", "start_seconds", "end_seconds", "speech_seconds", "quality_flags"]])
for record in records[:PREVIEW_CLIPS]:
    print(record["observation_id"], record["quality_flags"])
    display(Audio(filename=str(bundle_root / record["audio_file"])))

In [ ]:
archive_path: Path = run_root / f"{SERIES_ID}_{EPISODE_ID}_{CHARACTER_ID}_extraction.zip"
included_files: list[Path] = [bundle_root / record["audio_file"] for record in records]
included_files += [bundle_root / name for name in [
    "manifest.json", "segments.json", "diarization.json",
    "profile/speaker_profile.json", "review/speaker_matches.csv",
]]
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in included_files:
        archive.write(path, arcname=str(Path("extraction") / path.relative_to(bundle_root)))
print(f"Exported {len(records)} clips; ZIP size: {archive_path.stat().st_size / 1024**2:.1f} MiB")
files.download(str(archive_path))

# Save this standalone file for the next episode (also included inside the ZIP).
files.download(str(bundle_root / "profile" / "speaker_profile.json"))